# 作业 02：根据三个线索，自己完成训练

请参考作业 01，独立编写完整的 PyTorch 训练流程，让下面的预测函数尽量拟合给定数据。

本题只给出三个线索。参数如何创建、如何计算梯度以及如何更新，都由你完成。可以查阅作业 01，但请根据本题的预测函数调整代码。

## 线索 1：预测函数

这次有三个待学习参数 $w_1,w_2,w_3$：

$$\hat y=w_1x+w_2x^2+w_3.$$

请根据这个公式，自己编写计算预测值的代码。输入整个 `x` 张量时，应得到每个样本对应的预测值。三个待学习参数也由你创建。请保留这个预测公式。

其中，$w_1$ 是一次项的系数，$w_2$ 是二次项的系数，$w_3$ 是常数项。三个参数在训练中都需要被调整。

## 线索 2：损失函数

每个样本的损失仍然是预测误差的绝对值：

$$L_i=|y_i-\hat y_i|.$$

本次有多个样本，请将所有样本的损失取平均，得到用于训练的一个数：

$$L=\frac{1}{N}\sum_{i=1}^{N}|y_i-\hat y_i|.$$

这里 $N$ 是样本数。请自己用 PyTorch 写出这个损失，不要换成平方误差。

## 线索 3：两个数据张量

`x[i]` 和 `y[i]` 是同一个样本的输入和目标。训练时保持这些数据不变，不要直接填写你猜测的参数作为答案。

In [ ]:
import torch

# 输入数据与对应的目标数据，均为浮点数
x = torch.tensor([-2.0, -1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0])
y = torch.tensor([-3.0, -1.375, 0.0, 1.125, 2.0, 2.625, 3.0, 3.125, 3.0])

## 你的作答区

请在下面编写完整训练代码，可以自行增加单元格。训练结束后，展示学到的三个参数、每个样本的预测值，以及最终的平均绝对误差。最终展示的预测值和损失应使用训练结束后的参数计算。

再用文字说明：你对作业 01 的训练代码做了哪些修改？你如何判断这次训练有效？不要求参数或误差恰好等于某个数，也不要求每一轮损失都下降。

In [2]:
# 在这里编写你的训练代码
import torch

# 数据和目标值保持不变
x = torch.tensor([-2.0, -1.5, -1.0, -0.5, 0.0,
                  0.5, 1.0, 1.5, 2.0])
y = torch.tensor([-3.0, -1.375, 0.0, 1.125, 2.0,
                  2.625, 3.0, 3.125, 3.0])

# 创建三个待学习参数
w1 = torch.tensor(1.0, requires_grad=True)
w2 = torch.tensor(1.0, requires_grad=True)
w3 = torch.tensor(1.0, requires_grad=True)

# 三个参数都要交给优化器
optimizer = torch.optim.SGD([w1, w2, w3], lr=0.05)

epochs = 3000

# 记录训练前的损失，方便和训练后比较
with torch.no_grad():
    initial_pred = w1 * x + w2 * x**2 + w3
    initial_loss = torch.abs(y - initial_pred).mean()

for epoch in range(1, epochs + 1):
    # 让学习率从 0.05 逐渐减小到接近 0
    lr = 0.05 * (1 - epoch / (epochs + 1))
    optimizer.param_groups[0]["lr"] = lr

    # 1. 清除上一轮梯度
    optimizer.zero_grad()

    # 2. 前向计算：每个样本都有一个预测值
    y_pred = w1 * x + w2 * x**2 + w3

    # 3. 计算所有样本的平均绝对误差
    loss = torch.abs(y - y_pred).mean()

    # 4. 反向传播，计算 w1、w2、w3 的梯度
    loss.backward()

    # 5. 更新三个参数
    optimizer.step()

# 使用训练结束后的参数重新计算预测值和损失
with torch.no_grad():
    final_pred = w1 * x + w2 * x**2 + w3
    final_loss = torch.abs(y - final_pred).mean()

print("初始平均损失：", initial_loss.item())
print(f"w1 = {w1.item():.6f}")
print(f"w2 = {w2.item():.6f}")
print(f"w3 = {w3.item():.6f}")
print("最终预测：", [round(v, 6) for v in final_pred.tolist()])
print(f"最终平均绝对误差：{final_loss.item():.8f}")

初始平均损失： 2.0
w1 = 1.499994
w2 = -0.499973
w3 = 2.000000
最终预测： [-2.999879, -1.37493, 3.3e-05, 1.125009, 2.0, 2.625004, 3.000021, 3.125053, 3.000098]
最终平均绝对误差：0.00004542


### 我的观察与说明

我在作业 01 的基础上主要做了以下修改：第一，把原来的一个参数 w 改成了三个参数 w1、w2、w3，并都设置 requires_grad=True；第二，按照题目公式，用 w1 * x + w2 * x**2 + w3 一次计算所有样本的预测值；第三，使用 torch.abs(y - y_pred).mean() 计算所有样本的平均绝对误差；第四，把三个参数一起传给优化器，并使用逐渐减小的学习率帮助训练稳定下来。
训练开始时的平均绝对损失约为 2.0，训练结束时约为 0.0000454，预测值也已经非常接近每个目标值。最终学到的参数约为 w1=1.5、w2=-0.5、w3=2，说明模型已经接近对应的二次函数。这里不要求每一轮损失都下降，因为使用绝对值和有限学习率时，训练过程可能出现小幅波动

## 提交

重启内核并从上到下运行全部单元格，确认代码可以完整运行。保存包含代码、输出和文字回答的本文件，再提交到自己的 GitHub 仓库。